In [ ]:
import sys
import os
import gc
from pathlib import Path

import pandas as pd

sys.path.append(os.path.abspath("../../../"))
PROJECT_ROOT = "../../../"

from preprocessing.EEGMotorMovement.preprocessing import (
    load_physionet_eegmmidb_data
)
from preprocessing.general.filtering import (
    apply_filters_to_dataset,
    bands
)
import preprocessing.general.feature_extraction as fe


# ============================================================
# Paths
# ============================================================

root_edf = os.path.join(
    PROJECT_ROOT,
    "Datasets/EEG Motor Movement/original/files/"
)

output_csv = Path(
    os.path.join(
        PROJECT_ROOT,
        "Datasets/EEG Motor Movement/processed/EEG_MM_features.csv"
    )
)


# ============================================================
# Channel configuration
# ============================================================

"""
selected_channels = [
    "Fz",
    "FC3",
    "FC1",
    "FCz",
    "FC2",
    "FC4",
    "C5",
    "C3",
    "C1",
    "Cz",
    "C2",
    "C4",
    "C6",
    "CP3",
    "CP1",
    "CPz",
    "CP2",
    "CP4",
    "P1",
    "Pz",
    "P2",
    "POz",
]
"""

# Select a specific electrode configuration.
selected_channels = [
    "C3",
    "Cz",
    "C4",
]

# Use this instead to keep all 22 BCI EEG channels:
# selected_channels = None


# ============================================================
# Feature configuration
# ============================================================

extract_config = {
    "mean": {"function": fe.extract_mean},
    "std": {"function": fe.extract_std},
    "mom": {"function": fe.extract_moments},
    "min": {"function": fe.extract_min},
    "max": {"function": fe.extract_max},
    "cov": {"function": fe.extract_covariance},
    "eig": {"function": fe.extract_eigenvalues},

    # "logcov": {
    #     "function": fe.extract_logcov
    # },

    # "fft": {
    #     "function": fe.extract_fft,
    #     "params": {"ntop": 5},
    # },

    "h_diff": {"function": fe.extract_halves_diff},
    "q_stats": {"function": fe.extract_quarters_stats},
    "logvar": {"function": fe.extract_logvar},
}


# ============================================================
# Incremental processing configuration
# ============================================================

subjects = list(range(1, 110))

# Reduce to 1 for minimum memory usage.
SUBJECT_BATCH_SIZE = 5


# ============================================================
# Prepare output
# ============================================================

output_csv.parent.mkdir(
    parents=True,
    exist_ok=True,
)

# Prevent results from being appended to an old file.
if output_csv.exists():
    output_csv.unlink()

first_write = True
total_rows = 0


# ============================================================
# Load, filter, extract, and save incrementally
# ============================================================

for start in range(0, len(subjects), SUBJECT_BATCH_SIZE):

    subject_batch = subjects[
        start:start + SUBJECT_BATCH_SIZE
    ]

    print(
        f"\nProcessing subjects "
        f"{subject_batch[0]}–{subject_batch[-1]}"
    )

    # --------------------------------------------------------
    # Load current batch and select electrodes
    # --------------------------------------------------------

    batch_data = load_physionet_eegmmidb_data(
        root_dir=root_edf,
        config={
            "subjects": subject_batch,
            "channels": selected_channels,
        },
    )

    if not batch_data:
        print("⚠️ No data loaded for this batch.")
        continue

    print("✅ Data loading complete.")

    # --------------------------------------------------------
    # Filtering and resampling
    # --------------------------------------------------------

    filtered_data = apply_filters_to_dataset(
        dataset=batch_data,
        config={
            "original_fs": 160,
        },
    )

    print("✅ Filtering complete.")

    # --------------------------------------------------------
    # Feature extraction
    # --------------------------------------------------------

    df_batch = fe.extract_features_to_dataframe(
        dataset=filtered_data,
        extract_config=extract_config,
        band_labels=bands,
    )

    if df_batch.empty:
        print("⚠️ No features generated for this batch.")

        del batch_data, filtered_data, df_batch
        gc.collect()
        continue

    print(
        f"✅ Feature extraction complete: "
        f"{df_batch.shape}"
    )

    # --------------------------------------------------------
    # Append batch to CSV
    # --------------------------------------------------------

    df_batch.to_csv(
        output_csv,
        mode="w" if first_write else "a",
        header=first_write,
        index=False,
    )

    if first_write:
        display(df_batch.head())
        first_write = False

    total_rows += len(df_batch)

    print(
        f"✅ Batch saved. "
        f"Total rows written: {total_rows}"
    )

    # --------------------------------------------------------
    # Release memory
    # --------------------------------------------------------

    del batch_data
    del filtered_data
    del df_batch

    gc.collect()


# ============================================================
# Final report
# ============================================================

if first_write:
    print("⚠️ No feature data were written.")
else:
    print("\n✅ Complete feature extraction finished.")
    print(f"✅ Total rows: {total_rows}")
    print(f"✅ Saved to: {output_csv}")

/Users/edsonodake/miniforge3/envs/svm_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Processing subjects 1–5


Loading PhysioNet EEGMMIDB: 100%|██████████| 5/5 [00:01<00:00,  3.60subject/s]


✅ Data loading complete.
✅ Filtering complete.


Subjects: 100%|██████████| 5/5 [00:02<00:00,  1.69it/s]


...

✅ Feature extraction complete: (447, 5827)
✅ Batch saved. Total rows written: 9477

Processing subjects 106–109


Loading PhysioNet EEGMMIDB: 100%|██████████| 4/4 [00:00<00:00,  5.48subject/s]


✅ Data loading complete.
✅ Filtering complete.


Subjects: 100%|██████████| 4/4 [00:02<00:00,  1.86it/s]


✅ Feature extraction complete: (360, 5827)
✅ Batch saved. Total rows written: 9837

✅ Complete feature extraction finished.
✅ Total rows: 9837
✅ Saved to: ../../../Datasets/EEG Motor Movement/processed/EEG_MM_features.csv
